In [ ]:
"""
Module 1: Schema Design with Pydantic

This module implements Pydantic models for extracting structured customer feedback
from unstructured text. It demonstrates proper schema design with validation.
"""

from typing import Optional
from pydantic import BaseModel, Field, validator
import json


class CustomerFeedback(BaseModel):
    """
    Schema for extracting customer feedback information.

    - name: str (required, customer's full name)
    - email: Optional[str] (optional, validated email address)
    - product: str (required, product name)
    - issue_category: str (required, type of issue)
    - priority: str (required, must be: low, medium, high, or critical)
    - sentiment: float (required, score between -1.0 and 1.0)
    - feedback_text: str (required, the actual feedback content)

    After adding fields, implement validators for email, priority, and sentiment.
    See solution/ directory for examples.
    """
    name: str = Field(description="Customer's full name")
    email: Optional[str] = Field(default=None, description="Customer's email address")
    product: str = Field(description="Product name")
    issue_category: str = Field(description="Type of issue")
    priority: str = Field(description="Issue priority")
    sentiment: float = Field(description="Sentiment score", ge=-1.0, le=1.0)
    feedback_text: str = Field(description="Feedback content")

    @validator("email")
    def validate_email(cls, v):
        if v and "@" not in v:
            raise ValueError("Invalid email address")
        return v

    @validator("priority")
    def validate_priority(cls, v):
        if v not in ["low", "medium", "high", "critical"]:
            raise ValueError("Invalid priority")
        return v

    @validator("sentiment")
    def validate_sentiment(cls, v):
        if v < -1.0 or v > 1.0:
            raise ValueError("Sentiment must be between -1.0 and 1.0")
        return v


def extract_feedback_structured(feedback_dict: dict) -> CustomerFeedback:
    """
    Extract and validate customer feedback from a dictionary.
    """
    return CustomerFeedback(**feedback_dict)


def parse_feedback_from_json(json_string: str) -> CustomerFeedback:
    """
    Parse customer feedback from a JSON string.
    """
    try:
        feedback_dict = json.loads(json_string)
        return extract_feedback_structured(feedback_dict)
    except json.JSONDecodeError:
        raise ValueError("Invalid json format")

In [ ]:
"""
Module 2: Format Generators

This module implements functions to generate reports in multiple formats:
CSV, Markdown, and XML. Each format serves different stakeholders and use cases.
"""

from typing import List, Dict, Any
import csv
import io
from xml.etree import ElementTree as ET
from xml.dom import minidom


def generate_csv_report(feedback_list: List[Dict[str, Any]]) -> str:
    """
    Generate a CSV report from a list of customer feedback.
    """
    output = io.StringIO()
    writer = csv.DictWriter(output, fieldnames=feedback_list[0].keys())
    writer.writeheader()
    for feedback in feedback_list:
        writer.writerow(feedback)
    return output.getvalue()


def generate_markdown_report(feedback_list: List[Dict[str, Any]]) -> str:
    """
    Generate a Markdown report from customer feedback.
    """
    output = ["# Customer Feedback Report"]
    for feedback in feedback_list:
        output.append(f"## {feedback['product']}")
        output.append(f"**Name:** {feedback['name']}")
        output.append(f"**Email:** {feedback['email']}")
        output.append(f"**Issue Category:** {feedback['issue_category']}")
        output.append(f"**Priority:** {feedback['priority']}")
        output.append(f"**Sentiment:** {feedback['sentiment']}")
        output.append(f"**Feedback:** {feedback['feedback_text']}")
    return "\n".join(output)


def generate_xml_report(feedback_list: List[Dict[str, Any]]) -> str:
    """
    Generate an XML report from customer feedback.
    """
    root = ET.Element("CustomerFeedbackReport")
    for feedback in feedback_list:
        feedback_elem = ET.SubElement(root, "Feedback")
        for key, value in feedback.items():
            child = ET.SubElement(feedback_elem, key)
            child.text = str(value)
    xml_str = ET.tostring(root, encoding="utf-8")
    pretty_xml = minidom.parseString(xml_str).toprettyxml(indent="  ")
    return pretty_xml


def validate_csv_structure(csv_content: str) -> bool:
    """
    Validate that CSV content is well-formed.
    """
    output = io.StringIO(csv_content)
    reader = csv.DictReader(output)
    return all(reader.fieldnames) and all(row for row in reader)


def validate_xml_structure(xml_content: str) -> bool:
    """
    Validate that XML content is well-formed.
    """
    try:
        ET.fromstring(xml_content)
        return True
    except ET.ParseError:
        return False


In [ ]:
"""
Module 3: Multi-Layer Validation Pipeline

This module implements a production-grade validation system with multiple layers:
- Syntax validation (fast, catches format errors)
- Safety validation (detects PII and sensitive data)
- Semantic validation (checks business logic)
"""

import json
import re
import time
from typing import Dict, Any, Tuple, List


class ValidationResult:
    """Result of a validation check."""

    def __init__(self, is_valid: bool, layer: str, message: str, duration_ms: float):
        self.is_valid = is_valid
        self.layer = layer
        self.message = message
        self.duration_ms = duration_ms

    def __repr__(self):
        status = "✓" if self.is_valid else "✗"
        return f"{status} [{self.layer}] {self.message} ({self.duration_ms:.2f}ms)"


class ValidationLayer:
    """Base class for validation layers."""

    def __init__(self, name: str, timeout_ms: int):
        self.name = name
        self.timeout_ms = timeout_ms

    def validate(self, output: str) -> Tuple[bool, str]:
        """Validate output and return (is_valid, message)."""
        raise NotImplementedError


class SyntaxValidator(ValidationLayer):
    """Fast validation layer for JSON syntax and structure."""

    def __init__(self):
        super().__init__("syntax", 50)

    def validate(self, output: str) -> Tuple[bool, str]:
        """
        Check if output is valid JSON and has required fields.
        """
        try:
            data = json.loads(output)
            required_fields = ["name", "email", "product", "issue_category", "priority", "sentiment", "feedback_text"]
            for field in required_fields:
                if field not in data:
                    return False, f"Missing required field: {field}"
            return True, "Valid JSON"
        except json.JSONDecodeError:
            return False, "Invalid JSON"


class SafetyValidator(ValidationLayer):
    """Safety validation layer to detect PII and sensitive information."""

    def __init__(self):
        super().__init__("safety", 200)
        self.pii_patterns = {
            'ssn': r'\b\d{3}-\d{2}-\d{4}\b',
            'credit_card': r'\b\d{4}[\s-]?\d{4}[\s-]?\d{4}[\s-]?\d{4}\b',
            'phone': r'\b\d{3}[-.]?\d{3}[-.]?\d{4}\b'
        }

    def validate(self, output: str) -> Tuple[bool, str]:
        """
        Check output against PII patterns.
        """
        for field, pattern in self.pii_patterns.items():
            if re.search(pattern, output):
                return False, f"Detected PII in field: {field}"
        return True, "No PII detected"


class SemanticValidator(ValidationLayer):
    """Semantic validation layer for business logic and consistency."""

    def __init__(self):
        super().__init__("semantic", 100)

    def validate(self, output: str) -> Tuple[bool, str]:
        """
        Check priority-sentiment alignment.
        """
        try:
            data = json.loads(output)
            if data["priority"] == "critical" and data["sentiment"] > 0:
                return False, "Critical priority feedback must have negative sentiment."
            return True, "Valid semantic structure"
        except json.JSONDecodeError:
            return False, "Invalid JSON"


def validate_output_pipeline(output: str, layers: List[ValidationLayer]) -> List[ValidationResult]:
    """
    Run output through multi-layer validation pipeline.

    Run each layer and collect results with timing.
    """
    results = []
    for layer in layers:
        start_time = time.monotonic()
        is_valid, message = layer.validate(output)
        duration = (time.monotonic() - start_time) * 1000  # Convert to milliseconds
        results.append(ValidationResult(is_valid, layer.name, message, duration))
    return results


def recover_from_json_error(malformed_json: str) -> Dict[str, Any]:
    """
    Attempt to recover data from malformed JSON using vanilla Python.
    Handles trailing commas and missing closing braces.
    """
    if not malformed_json:
        return {}

    # 1. Try loading it cleanly first
    try:
        return json.loads(malformed_json)
    except json.JSONDecodeError:
        pass

    # 2. Clean up trailing whitespace/newlines
    cleaned = malformed_json.strip()

    # 3. Handle trailing comma inside an object
    # e.g., '{"a": 1,}' -> '{"a": 1}'
    if cleaned.endswith(',}') or cleaned.endswith(',\n}'):
        # Find the last comma before the closing brace and remove it
        r_index = cleaned.rfind(',')
        cleaned = cleaned[:r_index] + cleaned[r_index + 1:]

    # 4. Handle a trailing comma where the brace is entirely missing
    # e.g., '{"a": 1,' -> '{"a": 1}'
    elif cleaned.endswith(','):
        cleaned = cleaned.rstrip(',')

    # 5. Fix missing closing braces/brackets
    # Count open vs closed to see if we need to cap it off
    open_braces = cleaned.count('{')
    close_braces = cleaned.count('}')

    if open_braces > close_braces:
        cleaned += '}' * (open_braces - close_braces)

    # 6. Final attempt to parse the repaired string
    try:
        return json.loads(cleaned)
    except json.JSONDecodeError:
        return {}
